# 06 — Artefakt Export (Adim 10/11)

`models/metadata.json` (contract) ve `models/model_evidence.json` (rapor §10.1) JSON'lari uretilir, top-15 feature importance bar chart'lari ile gorsel sanity-check yapilir.

Cikti dosyalari:
* `models/metadata.json` (commit edilir)
* `models/model_evidence.json` (commit edilir)

In [ ]:
from __future__ import annotations
import sys, json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

In [ ]:
from ml.src.evidence import build_metadata, build_model_evidence

ev = build_model_evidence()
md = build_metadata()

MODELS_DIR = REPO_ROOT / 'models'
(MODELS_DIR / 'metadata.json').write_text(json.dumps(md, indent=2, ensure_ascii=False), encoding='utf-8')
(MODELS_DIR / 'model_evidence.json').write_text(json.dumps(ev, indent=2, ensure_ascii=False), encoding='utf-8')
print('classes (model_route.classes_):', len(md['models']['route']['classes']))
print('missing routes (15 - trained):', md['models']['route']['missing_classes_in_trained_model'])
print('profit feature count:', md['models']['profit']['n_features'])
print('honesty_note:', ev['honesty_note'])

In [ ]:
fi_profit = ev['feature_importance']['model_profit']
names = [f['feature'] for f in fi_profit]
vals = [f['importance'] for f in fi_profit]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(names[::-1], vals[::-1], color='#1f77b4')
ax.set_xlabel('CatBoost feature importance')
ax.set_title('Top-15 — model_profit')
plt.tight_layout()
plt.show()

In [ ]:
fi_route = ev['feature_importance']['model_route']
names = [f['feature'] for f in fi_route]
vals = [f['importance'] for f in fi_route]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(names[::-1], vals[::-1], color='#2ca02c')
ax.set_xlabel('CatBoost feature importance')
ax.set_title('Top-15 — model_route')
plt.tight_layout()
plt.show()

In [ ]:
print('--- profit_regression ---')
for k, v in ev['profit_regression'].items():
    if isinstance(v, dict):
        rmse = v.get('rmse')
        r2 = v.get('r2')
        print(f'  {k}: rmse={rmse:,.0f}  r2={r2:.3f}' if rmse is not None else f'  {k}: {v}')

print('\n--- route_classification ---')
for k, v in ev['route_classification'].items():
    if isinstance(v, dict):
        acc = v.get('accuracy')
        f1 = v.get('macro_f1')
        print(f'  {k}: acc={acc:.3f}  macro_f1={f1:.3f}' if acc is not None else f'  {k}: {v}')

print('\n--- ablation summary ---')
for k, v in ev['ablation'].items():
    print(f'  {k}: rmse_delta={v["rmse_delta_pct"]:+.1f}%')